In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

## Import DB, Its Connection and Integration


In [6]:
conn = sqlite3.connect('customer_churn.db')

sql_query = """
select name
from sqlite_master
where type='table'
"""

tables = pd.read_sql(sql_query, conn)

# create a df for each table
for table_name in tables['name']:
    df = pd.read_sql(f"select * from {table_name}", conn)
    globals()[f"df_{table_name}"] = df
    print(f"DataFrame for table '{table_name}' created with shape: {df.shape}")

conn.close()

DataFrame for table 'db_customer' created with shape: (21, 8)
DataFrame for table 'db_subscription' created with shape: (21, 11)
DataFrame for table 'db_support' created with shape: (9, 6)


In [8]:
conn = sqlite3.connect('customer_churn.db')

for table_name in tables['name']:
    print(f"DataFrame for table '{table_name}':")
    # Get column information
    columns_query = f"PRAGMA table_info({table_name})"
    columns = pd.read_sql(columns_query, conn)
    print(columns)
    print()
    print(columns['name'].tolist())
    print()

conn.close()

DataFrame for table 'db_customer':
   cid        name       type  notnull dflt_value  pk
0    0  customerid       TEXT        0       None   0
1    1        name       TEXT        0       None   0
2    2     country       TEXT        0       None   0
3    3       state       TEXT        0       None   0
4    4      gender       TEXT        0       None   0
5    5         dob  TIMESTAMP        0       None   0
6    6   interests       TEXT        0       None   0
7    7     pincode       REAL        0       None   0

['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode']

DataFrame for table 'db_subscription':
    cid                     name     type  notnull dflt_value  pk
0     0               customerid     TEXT        0       None   0
1     1  subscription_start_date     TEXT        0       None   0
2     2        subscription_type     TEXT        0       None   0
3     3             renewal_date     TEXT        0       None   0
4     4                p

In [9]:
df_db_customer.head()

,customerid,name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,None
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,NaN,None
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,None
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00,NaN,None
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00,drama,None


In [10]:
df_db_subscription.head()


,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaN,NaN,13.99,627,12
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaN,NaN,6.99,210,34
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaN,NaN,22.99,1725,8
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,88


In [11]:
df_db_support.head()

,customerid,complaint_date,escalations,csat_score,col_1,comment
0,0003-MKNFE,2024-08-28 00:00:00,N,60,None,service issue
1,0003-MKNFE,2024-08-28 00:00:00,Y,10,None,demaned refund
2,0013-EXCHZ,2024-01-20 00:00:00,Y,20,None,NaN
3,0013-MHZWF,2025-03-18 00:00:00,N,90,None,guidance to renew
4,0013-SMEOE,2024-11-01 00:00:00,N,30,None,NaN


# Data Cleaning

In [12]:
df_db_customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerid  21 non-null     str   
 1   name        21 non-null     str   
 2   country     18 non-null     str   
 3   state       21 non-null     str   
 4   gender      21 non-null     str   
 5   dob         21 non-null     str   
 6   interests   4 non-null      str   
 7   pincode     0 non-null      object
dtypes: object(1), str(7)
memory usage: 1.4+ KB


In [16]:
df_db_customer.tail()

,customerid,name,country,state,gender,dob,interests,pincode
16,0020-JDNXP,rikim,India,Meghalaya,Female,1994-08-19 00:00:00,NaN,None
17,0021-IKXGC,vishakha,India,Rajasthan,Female,2000-09-02 00:00:00,NaN,None
18,0022-TCJCI,raghvendra,India,Telangana,Male,1983-12-30 00:00:00,NaN,None
19,0023-HGHWL,rishabh,India,Uttar Pradesh,Men,1991-05-14 00:00:00,NaN,None
20,0023-UYUPN,sudevi,India,Maharashtra,Women,1977-10-06 00:00:00,NaN,None


In [17]:
# a. rename column - name
# b. drop_columns - interest and pincode
# c. change data type of column - date_of_birth to datetime
# d. data standardization - gender
# e. fix missing values - country



In [18]:
help(pd.DataFrame.rename)

Help on function rename in module pandas.core.frame:

rename(
    self,
    mapper: Renamer | None = None,
    *,
    index: Renamer | None = None,
    columns: Renamer | None = None,
    axis: Axis | None = None,
    copy: bool | lib.NoDefault = <no_default>,
    inplace: bool = False,
    level: Level | None = None,
    errors: IgnoreRaise = 'ignore'
) -> DataFrame | None
    Rename columns or index labels.

    Function / dict values must be unique (1-to-1). Labels not contained in
    a dict / Series will be left as-is. Extra labels listed don't throw an
    error.

    See the :ref:`user guide <basics.rename>` for more.

    Parameters
    ----------
    mapper : dict-like or function
        Dict-like or function transformations to apply to
        that axis' values. Use either ``mapper`` and ``axis`` to
        specify the axis to target with ``mapper``, or ``index`` and
        ``columns``.
    index : dict-like or function
        Alternative to specifying axis (``mapper, axis

In [19]:
# a. rename column - name
df_db_customer.rename?


Signature:
df_db_customer.rename(
    mapper: 'Renamer | None' = None,
    *,
    index: 'Renamer | None' = None,
    columns: 'Renamer | None' = None,
    axis: 'Axis | None' = None,
    copy: 'bool | lib.NoDefault' = <no_default>,
    inplace: 'bool' = False,
    level: 'Level | None' = None,
    errors: 'IgnoreRaise' = 'ignore',
) -> 'DataFrame | None'
Docstring:
Rename columns or index labels.

Function / dict values must be unique (1-to-1). Labels not contained in
a dict / Series will be left as-is. Extra labels listed don't throw an
error.

See the :ref:`user guide <basics.rename>` for more.

Parameters
----------
mapper : dict-like or function
    Dict-like or function transformations to apply to
    that axis' values. Use either ``mapper`` and ``axis`` to
    specify the axis to target with ``mapper``, or ``index`` and
    ``columns``.
index : dict-like or function
    Alternative to specifying axis (``mapper, axis=0``
    is equivalent to ``index=mapper``).
columns : dict-like

In [20]:
df_db_customer.rename(columns={'name': 'customer_name'}, inplace=True)